In [4]:
import h5py
import numpy as np

# Point this to the newly generated chain file in your MCMC_Chains folder
chain_path = "MCMC_Chains/Test_run/f1CDM_v/CC/CC_zeus.h5" 

with h5py.File(chain_path, "r") as f:
    print(f"--- Checking {chain_path} ---")
    print(f"Datasets found: {list(f.keys())}")
    
    if "log_like" in f:
        log_like = np.array(f["log_like"])
        log_prob = np.array(f["log_prob"])
        samples = np.array(f["samples"]) # or however your chain is named
        
        print(f"\nShapes:")
        print(f"Samples:   {samples.shape}")
        print(f"log_prob:  {log_prob.shape}")
        print(f"log_like:  {log_like.shape}")
        
        print(f"\nFirst 3 log_like values: {log_like[:3]}")
    else:
        print("\n❌ 'log_like' dataset missing!")

--- Checking MCMC_Chains/Test_run/f1CDM_v/CC/CC_zeus.h5 ---
Datasets found: ['log_prob', 'samples']

❌ 'log_like' dataset missing!


In [8]:
#Emcee test
import h5py
import numpy as np
import os
import glob

# Find the most recently modified .h5 file that does NOT have 'zeus' in the name
h5_files = [f for f in glob.glob("MCMC_Chains/**/*.h5", recursive=True) if "zeus" not in f.lower()]

if not h5_files:
    print("❌ No emcee .h5 files found. Run a model using emcee first!")
else:
    latest_emcee = max(h5_files, key=os.path.getmtime)
    print(f"✅ Found latest emcee chain: {latest_emcee}")
    
    with h5py.File(latest_emcee, "r") as f:
        if "log_like" not in f or "log_prob" not in f:
            print("❌ Chain is missing either log_prob or log_like.")
        else:
            log_prob = np.array(f["log_prob"])
            log_like = np.array(f["log_like"])
            
            print(f"Shape of log_prob: {log_prob.shape}")
            print(f"Shape of log_like: {log_like.shape}")
            
            # Check the maximum absolute difference between the two arrays
            diff = np.abs(log_prob - log_like)
            max_diff = np.max(diff)
            
            print(f"\n--- RESULTS ---")
            print(f"Maximum difference between log_prob and log_like: {max_diff}")
            
            if max_diff == 0.0:
                print("🎉 Claude is 100% correct! log_prob == log_like exactly in your codebase.")
                print("We can safely use his 1-line fix for Zeus.")
            else:
                print("⚠️ WARNING: There is a difference! Your priors are NOT exactly zero.")
                print(f"First 5 log_prob: {log_prob[:5]}")
                print(f"First 5 log_like: {log_like[:5]}")
                print("We CANNOT use the shortcut and must calculate true likelihoods separately.")

✅ Found latest emcee chain: MCMC_Chains/Test_run/f1CDM_v/BBN_PryMordial_DESI_DR2/BBN_PryMordial+DESI_DR2.h5
Shape of log_prob: (19200,)
Shape of log_like: (19200,)

--- RESULTS ---
Maximum difference between log_prob and log_like: 0.0
🎉 Claude is 100% correct! log_prob == log_like exactly in your codebase.
We can safely use his 1-line fix for Zeus.


In [6]:
import h5py
with h5py.File("MCMC_Chains/Test_run/f1CDM_v/CC/CC_zeus.h5", "r") as f:
    print(list(f.keys()))

['log_prob', 'samples']


In [ ]:
#!/usr/bin/env python3
"""
Read-only diagnostic script for the classy / clik import problems.

This script does NOT install, uninstall, modify, delete, or write to
anything on your system. It only inspects the current environment and
prints what it finds. Safe to run as many times as you like.

Usage:
    python kosmulator_diagnose.py
"""

import sys
import os
import glob
import importlib.util

def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def try_import(modname):
    try:
        mod = __import__(modname)
        path = getattr(mod, "__file__", "(no __file__ — built-in or namespace pkg)")
        return True, path
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


section("1) Python interpreter being used right now")
print("sys.executable:", sys.executable)
print("sys.version:", sys.version)

section("2) PYTHONPATH environment variable (as seen by THIS interpreter)")
pp = os.environ.get("PYTHONPATH", "")
if pp:
    for i, p in enumerate(pp.split(os.pathsep)):
        print(f"  [{i}] {p}")
else:
    print("  (not set)")

section("3) Full sys.path (what Python actually searches, in order)")
for i, p in enumerate(sys.path):
    print(f"  [{i}] {p}")

section("4) Attempting import of 'classy'")
ok, info = try_import("classy")
print("SUCCESS" if ok else "FAILED", "->", info)

section("5) Attempting import of 'clik'")
ok, info = try_import("clik")
print("SUCCESS" if ok else "FAILED", "->", info)

section("6) Searching for compiled classy .so files under $HOME (read-only glob, no /)")
home = os.path.expanduser("~")
hits = glob.glob(os.path.join(home, "**", "_classy*.so"), recursive=True)
if hits:
    for h in hits:
        print(" ", h)
else:
    print("  (none found under", home, ")")

section("7) Searching for clik package under $HOME (read-only glob, no /)")
hits = glob.glob(os.path.join(home, "**", "site-packages", "clik", "__init__.py"), recursive=True)
if hits:
    for h in hits:
        print(" ", h)
else:
    print("  (none found under", home, ")")

section("8) Searching for Kosmulator's own CLASS cache directory")
cache_dir = os.path.join(home, "Kosmulator", "Class")
if os.path.isdir(cache_dir):
    print("  Found:", cache_dir)
    for entry in sorted(os.listdir(cache_dir)):
        full = os.path.join(cache_dir, entry)
        print("   -", entry, "(dir)" if os.path.isdir(full) else "(file)")
else:
    print("  Not found at:", cache_dir)

section("9) Conda / environment info")
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV", "(not set)"))
print("CONDA_PREFIX:", os.environ.get("CONDA_PREFIX", "(not set)"))

section("DONE")
print("Copy everything above (from '1)' to here) and send it back.")
print("This script changed nothing on your system.")


1) Python interpreter being used right now
sys.executable: /usr/bin/python3
sys.version: 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]

2) PYTHONPATH environment variable (as seen by THIS interpreter)
  [0] /home/user/Kosmulator_Planck/code/plc_3.0/plc-3.1/lib/python/site-packages
  [1] 
  [2] /home/user/Kosmulator_Planck/code/plc_3.0/plc-3.1/lib/python/site-packages
  [3] 

3) Full sys.path (what Python actually searches, in order)
  [0] 
  [1] /home/user/Kosmulator_Planck/code/plc_3.0/plc-3.1/lib/python/site-packages
  [2] /home/user/Kosmulator
  [3] /usr/lib/python312.zip
  [4] /usr/lib/python3.12
  [5] /usr/lib/python3.12/lib-dynload
  [6] /home/user/.local/lib/python3.12/site-packages
  [7] /usr/local/lib/python3.12/dist-packages
  [8] /usr/lib/python3/dist-packages

4) Attempting import of 'classy'
FAILED -> ModuleNotFoundError: No module named 'classy'

5) Attempting import of 'clik'
Cannot use clik wrapper (cause = 'No module named 'clik.lkl'')
Cannot use clik_lensing wrap

In [3]:


import numpy as np

cov_path = "Observations/DESY5_covsys_000.txt"

# 1. Load and reshape
raw_data = np.loadtxt(cov_path)
n_dim = int(raw_data[0])
cov_loaded = raw_data[1:].reshape((n_dim, n_dim))
cov_loaded = 0.5 * (cov_loaded + cov_loaded.T)

# 2. Check shape and inversion sanity
print(f"Shape: {cov_loaded.shape}")
print(f"First 5 diagonal elements: {np.diag(cov_loaded)[:5]}")

# 3. Test inversion (mocking statistical error if data_sne isn't loaded yet)
mock_stat = np.ones(n_dim) * 0.1
cov_total = cov_loaded + np.diag(mock_stat**2)
inv_cov = np.linalg.inv(cov_total)

print("Inversion successful! Matrix is non-singular.")

Shape: (1829, 1829)
First 5 diagonal elements: [0.00019319 0.00853043 0.0027202  0.00053827 0.00153153]
Inversion successful! Matrix is non-singular.


In [2]:
from Kosmulator_main import Config
import User_defined_modules as UDM
from Kosmulator_main import constants as K

models = UDM.Get_model_names(["LCDM_v"])

prior_limits = {
    "Omega_m": (0.01, 1.0),
    "H_0": (40.0, 100.0),
    "r_d": (0.01, 1000.0),
    "M_abs": (-30.0, -5.0),
    # ... include whatever else your true DESY5/DESI setup needs;
    # copying the full dict from Kosmulator.py is safest
}

true_values = {
    "Omega_m": 0.315,
    "H_0": 67.4,
    "r_d": K.R_D_SINGLETON,
    "M_abs": -19.2,
}

CONFIG, data = Config.create_config(
    models=models,
    true_values=true_values,
    prior_limits=prior_limits,
    restrictions=UDM.Get_model_restrictions(["LCDM_v"]),
    observation=[["DESI_DR2", "DESY5"]],
    nwalkers=8,
    nsteps=100,
    burn=10,
    model_name=["LCDM_v"],
    logger=None,
)

print(list(data.keys()))  # confirm DESI_DR2 / DESY5 are in there

WARNING | Added ['H_0', 'r_d'] to parameters for ['DESI_DR2', 'DESY5'] in model LCDM_v


['CMB_lensing_RAW', 'CMB_lensing_CMBMARGED', 'DESI_DR2', 'DESY5']


In [5]:
import inspect
from Kosmulator_main import Statistical_packages as SP
print(inspect.getsource(SP.Calc_DESI_chi))

def Calc_DESI_chi(data, Model_func, param_dict, Type) -> float:
    """
    χ² for DESI VI-style BAO vectors with mixed observables.

    Supported type codes (after normalisation):
      3: D_V / r_d
      5: D_A / r_d
      6: D_H / r_d
      7: r_d / D_V
      8: D_M / r_d

    Any type==4 entries (raw D_V in Mpc) are auto-normalised to D_V / r_d
    and relabelled to 3, using a heuristic on the observed values.
    """
    def _must_calibrate(Type: str) -> bool:
        T = (Type or "").upper()
        return ("BBN" in T) or ("THETA" in T) or ("CMB" in T)

    z = np.asarray(data["redshift"], dtype=float)
    meas = np.asarray(data["measurement"], dtype=float).copy()
    types = np.asarray(data["type"], dtype=int).copy()
    cov = data.get("cov", None)
    inv_c = data.get("inv_cov", None)
    
    param_dict = _ensure_background_params(param_dict)

    DM = UDM.Comoving_distance_vectorized(Model_func, z, param_dict)  # Mpc
    Ez = Model_func(z, param_dict)
    DA = DM / (1.0 + z)

In [8]:
param_dict = {"Omega_m": 0.01, "H_0": 40.0, "r_d": 422.76}
chi2 = Calc_DESI_chi(data["DESI_DR2"], MODEL_func, param_dict, "DESI_DR2")
print("Wall-hugging point:", chi2)

sane_dict = {"Omega_m": 0.315, "H_0": 67.4, "r_d": 147.5}
chi2_sane = Calc_DESI_chi(data["DESI_DR2"], MODEL_func, sane_dict, "DESI_DR2")
print("Sane point:", chi2_sane)
sane_dict_desy5 = {"Omega_m": 0.315, "H_0": 67.4, "M_abs": -19.35}
# model_val needs computing first — Calc_Generic_SNe_chi takes a distance-modulus array, not param_dict alone

Wall-hugging point: 10531.308014428996
Sane point: 23.0879647474472


In [9]:
from Kosmulator_main.Statistical_packages import Calc_Generic_SNe_chi
import Kosmulator_main.utils as U

redshift = data["DESY5"]["redshift"]
comoving_distances = U.Comoving_distance_vectorized(MODEL_func, redshift, sane_dict_desy5)
model_val = 25 + 5 * np.log10(comoving_distances * (1 + redshift))

chi2_desy5 = Calc_Generic_SNe_chi(obs_data=data["DESY5"], model=model_val, param_dict=sane_dict_desy5)
print("DESY5 sane point:", chi2_desy5)

DESY5 sane point: 22005656.0717076


In [10]:
print("inv_cov" in data["DESY5"])
print("cov" in data["DESY5"])
if "inv_cov" in data["DESY5"]:
    ic = np.asarray(data["DESY5"]["inv_cov"])
    print("shape:", ic.shape)
    print("diagonal sample:", np.diag(ic)[:5])

True
True
shape: (1829, 1829)
diagonal sample: [106.16407949  53.77230367  18.32152107  44.20288861  38.06980482]


In [12]:
cov = np.asarray(data["DESY5"]["cov"])
inv_cov = np.asarray(data["DESY5"]["inv_cov"])

product = cov @ inv_cov
identity_check = np.allclose(product, np.eye(product.shape[0]), atol=1e-3)
print("cov @ inv_cov ≈ I ?", identity_check)
print("diagonal of product (should be ~1):", np.diag(product)[:5])
print("off-diagonal sample (should be ~0):", product[0, 1:5])

residual_test = np.zeros(cov.shape[0])  # dummy, just testing the matrix itself
manual_inv = np.linalg.inv(cov)
print("manual inverse diagonal sample:", np.diag(manual_inv)[:5])
print("stored inv_cov diagonal sample:", np.diag(inv_cov)[:5])

cov @ inv_cov ≈ I ? True
diagonal of product (should be ~1): [1. 1. 1. 1. 1.]
off-diagonal sample (should be ~0): [ 2.49366500e-18 -2.71050543e-19 -1.08420217e-19  1.68051337e-18]
manual inverse diagonal sample: [106.16407949  53.77230367  18.32152107  44.20288861  38.06980482]
stored inv_cov diagonal sample: [106.16407949  53.77230367  18.32152107  44.20288861  38.06980482]


In [14]:
type_data = np.asarray(data["DESY5"]["type_data"])
M = sane_dict_desy5.get("M_abs", -19.35)
residual = (type_data - M) - model_val

print("type_data range:", type_data.min(), type_data.max())
print("model_val range:", model_val.min(), model_val.max())
print("residual range:", residual.min(), residual.max())
print("residual mean/std:", residual.mean(), residual.std())

print("data_is_distance_modulus flag:", data["DESY5"].get("data_is_distance_modulus"))

type_data range: 34.9553 44.5882
model_val range: 35.19286803192841 44.46948065886076
residual range: 18.334227772577783 22.313040866068057
residual mean/std: 19.31293612315145 0.2603881184324923
data_is_distance_modulus flag: False


In [15]:
residual_correct = type_data - model_val
print("corrected residual mean/std:", residual_correct.mean(), residual_correct.std())

corrected residual mean/std: -0.03706387684855396 0.2603881184324923


In [16]:
inv_cov = data["DESY5"]["inv_cov"]
chi2_fixed = float(residual_correct @ inv_cov @ residual_correct)
print("DESY5 sane point, corrected:", chi2_fixed)

DESY5 sane point, corrected: 2046.6754379201648
